# LM2011 Refinitiv Finalize Helper (Colab)

This notebook finalizes the LM2011 document-level Refinitiv artifacts without rerunning the full `sec_ccm_unified_runner` notebook.

It handles only these focused outputs:

- `refinitiv_doc_ownership_lm2011/refinitiv_lm2011_doc_ownership.parquet`
- `refinitiv_doc_analyst_lm2011/refinitiv_doc_analyst_request_anchors.parquet`
- `refinitiv_doc_analyst_lm2011/refinitiv_doc_analyst_selected.parquet`

It expects the LSEG/API raw or normalized inputs to already exist in the Drive run root.

In [1]:
from google.colab import drive
import os
import subprocess
import sys
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

IN_COLAB = 'google.colab' in sys.modules
REPO_ENV_VAR = 'NLP_THESIS_REPO_ROOT'
GIT_URL_ENV_VAR = 'NLP_THESIS_GIT_URL'
DEFAULT_GIT_URL = 'https://github.com/Erew42/NLP_Thesis.git'
CLONE_ROOT = Path('/content/NLP_Thesis')


def _resolve_drive_root() -> Path:
    for candidate in (
        Path('/content/drive/MyDrive'),
        Path('/content/drive/My Drive'),
        Path('/content/drive'),
    ):
        if candidate.exists():
            return candidate
    return Path('/content/drive')


def _find_repo_root(drive_root: Path) -> Path | None:
    override = os.environ.get(REPO_ENV_VAR, '').strip()
    candidates: list[Path] = []
    if override:
        candidates.append(Path(override).expanduser())
    candidates.extend(
        [
            Path.cwd().resolve(),
            *Path.cwd().resolve().parents,
            CLONE_ROOT,
            drive_root / 'NLP_Thesis',
            drive_root / 'MyDrive' / 'NLP_Thesis',
            drive_root / 'My Drive' / 'NLP_Thesis',
        ]
    )
    seen: set[Path] = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / 'src' / 'thesis_pkg' / 'pipeline.py').exists():
            return candidate
    return None


DRIVE_ROOT = _resolve_drive_root()
REPO_ROOT = _find_repo_root(DRIVE_ROOT)
if REPO_ROOT is None and IN_COLAB:
    git_url = os.environ.get(GIT_URL_ENV_VAR, DEFAULT_GIT_URL)
    if CLONE_ROOT.exists() and not (CLONE_ROOT / 'src' / 'thesis_pkg' / 'pipeline.py').exists():
        raise FileExistsError(
            f'{CLONE_ROOT} exists but does not look like the NLP_Thesis repo. Remove it or set {REPO_ENV_VAR}.'
        )
    if not CLONE_ROOT.exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', git_url, str(CLONE_ROOT)])
    REPO_ROOT = CLONE_ROOT

if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate the NLP_Thesis checkout. Set NLP_THESIS_REPO_ROOT.')

os.environ[REPO_ENV_VAR] = str(REPO_ROOT)
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print({'DRIVE_ROOT': str(DRIVE_ROOT), 'REPO_ROOT': str(REPO_ROOT)})

Mounted at /content/drive
{'DRIVE_ROOT': '/content/drive/MyDrive', 'REPO_ROOT': '/content/NLP_Thesis'}


In [2]:
# The source checkout alone is not enough in a fresh Colab runtime. Install the
# small set of runtime dependencies imported by thesis_pkg's package surface.
REQUIRED_RUNTIME_PACKAGES = [
    'polars',
    'pyarrow',
    'pandas',
    'xlsxwriter',
    'openpyxl',
    'statsmodels',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIRED_RUNTIME_PACKAGES])
print({'installed_or_verified': REQUIRED_RUNTIME_PACKAGES})

{'installed_or_verified': ['polars', 'pyarrow', 'pandas', 'xlsxwriter', 'openpyxl', 'statsmodels']}


In [3]:
import polars as pl

from thesis_pkg.core.ccm.lm2011 import build_lm2011_sample_backbone, build_quarterly_accounting_panel
from thesis_pkg.notebooks_and_scripts.lm2011_sample_post_refinitiv_runner import _prepare_lm2011_sec_input_lf
from thesis_pkg.pipelines.refinitiv.analyst import (
    run_refinitiv_lm2011_doc_analyst_anchor_pipeline,
    run_refinitiv_lm2011_doc_analyst_select_pipeline,
)
from thesis_pkg.pipelines.refinitiv.doc_ownership import run_refinitiv_lm2011_doc_ownership_finalize_pipeline


def _env_path(name: str, default: Path) -> Path:
    value = os.environ.get(name, '').strip()
    return Path(value).expanduser() if value else default


WORK_ROOT = _env_path('LM2011_WORK_ROOT', DRIVE_ROOT / 'Data_LM')
RUN_ROOT = _env_path('LM2011_REFINITIV_RUN_ROOT', WORK_ROOT / 'results' / 'sec_ccm_unified_runner')
SEC_YEAR_MERGED_DIR = _env_path('LM2011_YEAR_MERGED_DIR', WORK_ROOT / 'parquet_data' / '_year_merged')
CCM_BASE_DIR = _env_path('LM2011_CCM_BASE_DIR', WORK_ROOT / 'CRSP_Compustat_data' / 'parquet_data')
SEC_CCM_PREMERGE_DIR = _env_path('LM2011_SEC_CCM_PREMERGE_DIR', RUN_ROOT / 'sec_ccm_premerge')
MATCHED_CLEAN_PATH = _env_path('LM2011_MATCHED_CLEAN_PATH', SEC_CCM_PREMERGE_DIR / 'sec_ccm_matched_clean.parquet')
LM2011_BACKBONE_PATH = _env_path('LM2011_BACKBONE_PATH', SEC_CCM_PREMERGE_DIR / 'lm2011_sample_backbone.parquet')
REFINITIV_DOC_OWNERSHIP_LM2011_DIR = _env_path(
    'LM2011_DOC_OWNERSHIP_DIR', RUN_ROOT / 'refinitiv_doc_ownership_lm2011'
)
REFINITIV_STEP1_DIR = _env_path('LM2011_REFINITIV_STEP1_DIR', RUN_ROOT / 'refinitiv_step1')
REFINITIV_ANALYST_COMMON_STOCK_DIR = _env_path(
    'LM2011_ANALYST_COMMON_STOCK_DIR', REFINITIV_STEP1_DIR / 'analyst_common_stock'
)
REFINITIV_DOC_ANALYST_LM2011_DIR = _env_path(
    'LM2011_DOC_ANALYST_DIR', RUN_ROOT / 'refinitiv_doc_analyst_lm2011'
)
FALLBACK_FILLED_WORKBOOK_PATH = _env_path(
    'LM2011_OWNERSHIP_FALLBACK_FILLED_WORKBOOK',
    REFINITIV_DOC_OWNERSHIP_LM2011_DIR / 'refinitiv_lm2011_doc_ownership_fallback_handoff_filled_in.xlsx',
)

FORCE_REBUILD_BACKBONE = False
RUN_DOC_OWNERSHIP_FINALIZE = True
RUN_DOC_ANALYST_ANCHORS = True
RUN_DOC_ANALYST_SELECT = True

print(
    {
        'WORK_ROOT': str(WORK_ROOT),
        'RUN_ROOT': str(RUN_ROOT),
        'SEC_YEAR_MERGED_DIR': str(SEC_YEAR_MERGED_DIR),
        'CCM_BASE_DIR': str(CCM_BASE_DIR),
        'MATCHED_CLEAN_PATH': str(MATCHED_CLEAN_PATH),
        'LM2011_BACKBONE_PATH': str(LM2011_BACKBONE_PATH),
        'REFINITIV_DOC_OWNERSHIP_LM2011_DIR': str(REFINITIV_DOC_OWNERSHIP_LM2011_DIR),
        'REFINITIV_ANALYST_COMMON_STOCK_DIR': str(REFINITIV_ANALYST_COMMON_STOCK_DIR),
        'REFINITIV_DOC_ANALYST_LM2011_DIR': str(REFINITIV_DOC_ANALYST_LM2011_DIR),
    }
)

{'WORK_ROOT': '/content/drive/MyDrive/Data_LM', 'RUN_ROOT': '/content/drive/MyDrive/Data_LM/results/sec_ccm_unified_runner', 'SEC_YEAR_MERGED_DIR': '/content/drive/MyDrive/Data_LM/parquet_data/_year_merged', 'CCM_BASE_DIR': '/content/drive/MyDrive/Data_LM/CRSP_Compustat_data/parquet_data', 'MATCHED_CLEAN_PATH': '/content/drive/MyDrive/Data_LM/results/sec_ccm_unified_runner/sec_ccm_premerge/sec_ccm_matched_clean.parquet', 'LM2011_BACKBONE_PATH': '/content/drive/MyDrive/Data_LM/results/sec_ccm_unified_runner/sec_ccm_premerge/lm2011_sample_backbone.parquet', 'REFINITIV_DOC_OWNERSHIP_LM2011_DIR': '/content/drive/MyDrive/Data_LM/results/sec_ccm_unified_runner/refinitiv_doc_ownership_lm2011', 'REFINITIV_ANALYST_COMMON_STOCK_DIR': '/content/drive/MyDrive/Data_LM/results/sec_ccm_unified_runner/refinitiv_step1/analyst_common_stock', 'REFINITIV_DOC_ANALYST_LM2011_DIR': '/content/drive/MyDrive/Data_LM/results/sec_ccm_unified_runner/refinitiv_doc_analyst_lm2011'}


In [4]:
def _first_existing_path(*paths: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError(f'No existing path found among: {[str(path) for path in paths]}')


def _resolve_ccm_parquet_artifact(base_dir: Path, parquet_name: str) -> Path:
    candidates = [base_dir / parquet_name]
    candidates.extend(sorted(child / parquet_name for child in base_dir.glob('documents-export*') if child.is_dir()))
    return _first_existing_path(*candidates)


def _row_count(path: Path) -> int | None:
    if not path.exists():
        return None
    return int(pl.scan_parquet(path).select(pl.len().alias('n')).collect().item())


def _path_status(paths: dict[str, Path]) -> pl.DataFrame:
    rows = []
    for label, path in paths.items():
        rows.append({'label': label, 'path': str(path), 'exists': path.exists()})
    return pl.DataFrame(rows)


def _require_paths(paths: dict[str, Path]) -> None:
    status = _path_status(paths)
    display(status)
    missing = status.filter(pl.col('exists').not_())
    if missing.height:
        raise FileNotFoundError('Missing required inputs:\n' + '\n'.join(missing.get_column('path').to_list()))


FILINGDATES_PATH = _resolve_ccm_parquet_artifact(CCM_BASE_DIR, 'filingdates.parquet')
QUARTERLY_BALANCE_PATH = _resolve_ccm_parquet_artifact(CCM_BASE_DIR, 'balancesheetquarterly.parquet')
QUARTERLY_INCOME_PATH = _resolve_ccm_parquet_artifact(CCM_BASE_DIR, 'incomestatementquarterly.parquet')
QUARTERLY_PERIOD_DESCRIPTOR_PATH = _resolve_ccm_parquet_artifact(CCM_BASE_DIR, 'perioddescriptorquarterly.parquet')
ANALYST_NORMALIZED_PANEL_PATH = REFINITIV_ANALYST_COMMON_STOCK_DIR / 'refinitiv_analyst_normalized_panel.parquet'

required_paths = {
    'SEC_YEAR_MERGED_DIR': SEC_YEAR_MERGED_DIR,
    'MATCHED_CLEAN_PATH': MATCHED_CLEAN_PATH,
    'FILINGDATES_PATH': FILINGDATES_PATH,
    'QUARTERLY_BALANCE_PATH': QUARTERLY_BALANCE_PATH,
    'QUARTERLY_INCOME_PATH': QUARTERLY_INCOME_PATH,
    'QUARTERLY_PERIOD_DESCRIPTOR_PATH': QUARTERLY_PERIOD_DESCRIPTOR_PATH,
}
if RUN_DOC_OWNERSHIP_FINALIZE:
    required_paths.update(
        {
            'OWNERSHIP_EXACT_REQUESTS': REFINITIV_DOC_OWNERSHIP_LM2011_DIR / 'refinitiv_lm2011_doc_ownership_exact_requests.parquet',
            'OWNERSHIP_EXACT_RAW': REFINITIV_DOC_OWNERSHIP_LM2011_DIR / 'refinitiv_lm2011_doc_ownership_exact_raw.parquet',
            'OWNERSHIP_FALLBACK_REQUESTS': REFINITIV_DOC_OWNERSHIP_LM2011_DIR / 'refinitiv_lm2011_doc_ownership_fallback_requests.parquet',
        }
    )
if RUN_DOC_ANALYST_SELECT:
    required_paths['ANALYST_NORMALIZED_PANEL_PATH'] = ANALYST_NORMALIZED_PANEL_PATH

_require_paths(required_paths)

label,path,exists
str,str,bool
"""SEC_YEAR_MERGED_DIR""","""/content/drive/MyDrive/Data_LM…",true
"""MATCHED_CLEAN_PATH""","""/content/drive/MyDrive/Data_LM…",true
"""FILINGDATES_PATH""","""/content/drive/MyDrive/Data_LM…",true
"""QUARTERLY_BALANCE_PATH""","""/content/drive/MyDrive/Data_LM…",true
"""QUARTERLY_INCOME_PATH""","""/content/drive/MyDrive/Data_LM…",true
"""QUARTERLY_PERIOD_DESCRIPTOR_PA…","""/content/drive/MyDrive/Data_LM…",true
"""OWNERSHIP_EXACT_REQUESTS""","""/content/drive/MyDrive/Data_LM…",true
"""OWNERSHIP_EXACT_RAW""","""/content/drive/MyDrive/Data_LM…",true
"""OWNERSHIP_FALLBACK_REQUESTS""","""/content/drive/MyDrive/Data_LM…",true


In [5]:
def build_or_reuse_lm2011_backbone(*, force: bool = False) -> Path:
    if LM2011_BACKBONE_PATH.exists() and not force:
        print({'using_existing_lm2011_backbone': str(LM2011_BACKBONE_PATH), 'rows': _row_count(LM2011_BACKBONE_PATH)})
        return LM2011_BACKBONE_PATH

    year_paths = sorted(path for path in SEC_YEAR_MERGED_DIR.glob('*.parquet') if path.is_file())
    if not year_paths:
        raise FileNotFoundError(f'No year parquet files found in {SEC_YEAR_MERGED_DIR}')

    first_schema = pl.scan_parquet(year_paths[0]).collect_schema()
    required_sec_cols = ['doc_id', 'cik_10', 'accession_nodash', 'file_date_filename', 'document_type_filename']
    missing = [name for name in required_sec_cols if name not in first_schema]
    if missing:
        raise ValueError(f'Year parquet schema missing required LM2011 backbone columns: {missing}')

    sec_lf = _prepare_lm2011_sec_input_lf(
        pl.scan_parquet([str(path) for path in year_paths]).select([pl.col(name) for name in required_sec_cols])
    )
    backbone_lf = build_lm2011_sample_backbone(
        sec_lf,
        pl.scan_parquet(MATCHED_CLEAN_PATH),
        ccm_filingdates_lf=pl.scan_parquet(FILINGDATES_PATH),
    )
    LM2011_BACKBONE_PATH.parent.mkdir(parents=True, exist_ok=True)
    backbone_lf.sink_parquet(LM2011_BACKBONE_PATH, compression='zstd')
    print({'wrote_lm2011_backbone': str(LM2011_BACKBONE_PATH), 'rows': _row_count(LM2011_BACKBONE_PATH)})
    return LM2011_BACKBONE_PATH


lm2011_backbone_path = build_or_reuse_lm2011_backbone(force=FORCE_REBUILD_BACKBONE)

{'wrote_lm2011_backbone': '/content/drive/MyDrive/Data_LM/results/sec_ccm_unified_runner/sec_ccm_premerge/lm2011_sample_backbone.parquet', 'rows': 60965}


In [6]:
ownership_outputs: dict[str, Path] = {}

if RUN_DOC_OWNERSHIP_FINALIZE:
    fallback_requests_path = REFINITIV_DOC_OWNERSHIP_LM2011_DIR / 'refinitiv_lm2011_doc_ownership_fallback_requests.parquet'
    fallback_raw_path = REFINITIV_DOC_OWNERSHIP_LM2011_DIR / 'refinitiv_lm2011_doc_ownership_fallback_raw.parquet'
    fallback_eligible_count = 0
    if fallback_requests_path.exists():
        fallback_eligible_count = int(
            pl.scan_parquet(fallback_requests_path)
            .filter(pl.col('retrieval_eligible').fill_null(False))
            .select(pl.len())
            .collect()
            .item()
        )
    fallback_workbook = (
        FALLBACK_FILLED_WORKBOOK_PATH
        if fallback_eligible_count and not fallback_raw_path.exists()
        else None
    )
    if fallback_workbook is not None and not fallback_workbook.exists():
        raise FileNotFoundError(
            f'Fallback ownership requests exist but neither fallback raw parquet nor filled workbook exists: {fallback_workbook}'
        )
    ownership_outputs = run_refinitiv_lm2011_doc_ownership_finalize_pipeline(
        output_dir=REFINITIV_DOC_OWNERSHIP_LM2011_DIR,
        fallback_filled_workbook_path=fallback_workbook,
    )
    display(
        pl.DataFrame(
            [
                {'artifact': key, 'path': str(path), 'rows': _row_count(path)}
                for key, path in ownership_outputs.items()
            ]
        )
    )
else:
    print('RUN_DOC_OWNERSHIP_FINALIZE is False; skipping ownership finalize.')

artifact,path,rows
str,str,i64
"""refinitiv_lm2011_doc_ownership…","""/content/drive/MyDrive/Data_LM…",0
"""refinitiv_lm2011_doc_ownership…","""/content/drive/MyDrive/Data_LM…",67723
"""refinitiv_lm2011_doc_ownership…","""/content/drive/MyDrive/Data_LM…",8807


In [7]:
analyst_outputs: dict[str, Path] = {}

quarterly_accounting_panel_lf = build_quarterly_accounting_panel(
    pl.scan_parquet(QUARTERLY_BALANCE_PATH),
    pl.scan_parquet(QUARTERLY_INCOME_PATH),
    pl.scan_parquet(QUARTERLY_PERIOD_DESCRIPTOR_PATH),
)

if RUN_DOC_ANALYST_ANCHORS:
    anchor_outputs = run_refinitiv_lm2011_doc_analyst_anchor_pipeline(
        sample_backbone_lf=pl.scan_parquet(lm2011_backbone_path),
        quarterly_accounting_panel_lf=quarterly_accounting_panel_lf,
        output_dir=REFINITIV_DOC_ANALYST_LM2011_DIR,
    )
    analyst_outputs.update(anchor_outputs)
    doc_anchors_path = anchor_outputs['refinitiv_doc_analyst_request_anchors_parquet']
else:
    doc_anchors_path = REFINITIV_DOC_ANALYST_LM2011_DIR / 'refinitiv_doc_analyst_request_anchors.parquet'

if RUN_DOC_ANALYST_SELECT:
    if not doc_anchors_path.exists():
        raise FileNotFoundError(f'Doc analyst anchors not found: {doc_anchors_path}')
    select_outputs = run_refinitiv_lm2011_doc_analyst_select_pipeline(
        doc_anchors_artifact_path=doc_anchors_path,
        analyst_normalized_panel_artifact_path=ANALYST_NORMALIZED_PANEL_PATH,
        output_dir=REFINITIV_DOC_ANALYST_LM2011_DIR,
    )
    analyst_outputs.update(select_outputs)

display(
    pl.DataFrame(
        [
            {'artifact': key, 'path': str(path), 'rows': _row_count(path)}
            for key, path in analyst_outputs.items()
        ]
    )
)

artifact,path,rows
str,str,i64
"""refinitiv_doc_analyst_request_…","""/content/drive/MyDrive/Data_LM…",60965
"""refinitiv_doc_analyst_selected…","""/content/drive/MyDrive/Data_LM…",79444


In [8]:
final_artifacts = {
    'lm2011_sample_backbone': lm2011_backbone_path,
    'refinitiv_lm2011_doc_ownership': REFINITIV_DOC_OWNERSHIP_LM2011_DIR / 'refinitiv_lm2011_doc_ownership.parquet',
    'refinitiv_doc_analyst_request_anchors': REFINITIV_DOC_ANALYST_LM2011_DIR / 'refinitiv_doc_analyst_request_anchors.parquet',
    'refinitiv_doc_analyst_selected': REFINITIV_DOC_ANALYST_LM2011_DIR / 'refinitiv_doc_analyst_selected.parquet',
}

summary = []
for label, path in final_artifacts.items():
    summary.append({'artifact': label, 'path': str(path), 'exists': path.exists(), 'rows': _row_count(path)})
display(pl.DataFrame(summary))

artifact,path,exists,rows
str,str,bool,i64
"""lm2011_sample_backbone""","""/content/drive/MyDrive/Data_LM…",true,60965
"""refinitiv_lm2011_doc_ownership""","""/content/drive/MyDrive/Data_LM…",true,8807
"""refinitiv_doc_analyst_request_…","""/content/drive/MyDrive/Data_LM…",true,60965
"""refinitiv_doc_analyst_selected""","""/content/drive/MyDrive/Data_LM…",true,79444
